# 🚒 London Fire Brigade — Analyse Exploratoire des Données
**Liora Data Analyst Bootcamp | Les Mines PSL | Février 2026**
**Auteur : Gaëtan Seynave**

---

## Contexte
Ce notebook présente l'analyse exploratoire des données de la London Fire Brigade (LFB)
sur la période **2021–2025** (post-COVID, années pleines).

Les données ont été préparées dans `01_preprocessing.ipynb`.

**Sources :** [London Data Store](https://data.london.gov.uk/)
**Dashboard Power BI :** [Voir le rapport interactif](https://app.powerbi.com/view?r=eyJrIjoiN2UyNTMxYTItZmY2Yy00OGQzLTk3ZjItOGFlZDRkNDY3NGE3IiwidCI6IjI3YThmZjFkLTc1YTAtNGFlNC1hNTg2LTMxN2NlOWVhYWZmYiJ9&pageName=4952a90622ed792a18d3)

---

## Problématique principale
> Comprendre la répartition et la nature des incidents à Londres afin d'évaluer
> l'efficacité de la réponse opérationnelle de la LFB et l'adéquation des ressources
> mobilisées face aux besoins du territoire.

---

## Structure du notebook

| Section | Contenu |
|---|---|
| 0 | Imports & chargement des données |
| 1 | Vue d'ensemble 2021–2025 |
| 2 | Analyse univariée — Incidents |
| 3 | Analyse univariée — Mobilisations |
| 4 | Analyse multivariée |
| 5 | Performance légale (objectifs 6 et 8 minutes) |

## 0. Imports & chargement des données

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
pd.set_option('display.max_columns', None)

# Palette cohérente pour tout le notebook
PALETTE_LEGAL = ['#ff9999', '#99cc99']  # Rouge = retard, Vert = dans les temps
PALETTE_CATEGORIES = 'Set2'

In [ ]:
dossier = os.path.join(os.getcwd(), 'data', 'processed')

incident    = pd.read_parquet(os.path.join(dossier, 'incident_powerbi_gaetan.parquet'))
mobilisation = pd.read_parquet(os.path.join(dossier, 'mobilisation_powerbi_gaetan.parquet'))

print(f"✅ Incidents     : {len(incident):,} lignes — {incident.shape[1]} colonnes")
print(f"✅ Mobilisations : {len(mobilisation):,} lignes — {mobilisation.shape[1]} colonnes")
print(f"   Période       : {incident['DateOfCall'].min().date()} → {incident['DateOfCall'].max().date()}")

## 1. Vue d'ensemble 2021–2025

### 1.1 Volume d'incidents, coûts et tendances annuelles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Graphique 1 : Nombre d'incidents par année
incident['CalYear'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', rot=0)
axes[0].set_title("Nombre d'incidents par année")
axes[0].set_xlabel("Année")
axes[0].set_ylabel("Nombre d'incidents")
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=9)

# Graphique 2 : Coût estimatif par année
cout_an = incident.groupby('CalYear')['Notional Cost (£)'].sum().reset_index()
axes[1].bar(cout_an['CalYear'], cout_an['Notional Cost (£)'] / 1e6, color='coral')
axes[1].set_title("Coût estimatif total par année (M£)")
axes[1].set_xlabel("Année")
axes[1].set_ylabel("Coût (millions £)")

# Graphique 3 : Coût moyen par incident
cout_moy = incident.groupby('CalYear')['Notional Cost (£)'].mean().reset_index()
axes[2].plot(cout_moy['CalYear'], cout_moy['Notional Cost (£)'],
             marker='o', color='darkorange', linewidth=2)
axes[2].set_title("Coût moyen par incident")
axes[2].set_xlabel("Année")
axes[2].set_ylabel("Coût moyen (£)")
axes[2].grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle("LFB 2021–2025 : Vue d'ensemble", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

> **Observation :** Le nombre d'incidents est en **hausse constante (+30% sur 5 ans)**
> tandis que le budget ne progresse que de +21%. Le coût alloué par incident diminue
> mécaniquement : le service est **sous pression croissante**.

## 2. Analyse univariée — Incidents

### 2.1 Types d'incidents

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Camembert
counts = incident['IncidentGroup'].value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=sns.color_palette(PALETTE_CATEGORIES, len(counts)),
            startangle=90)
axes[0].set_title("Répartition des types d'incidents (2021–2025)")

# Évolution annuelle
nb_incident = incident.groupby(['CalYear', 'IncidentGroup']).size().reset_index(name='count')
sns.lineplot(data=nb_incident, x='CalYear', y='count',
             hue='IncidentGroup', marker='o', ax=axes[1],
             palette=PALETTE_CATEGORIES)
axes[1].set_title("Évolution du volume par type d'incident")
axes[1].set_xlabel("Année")
axes[1].set_ylabel("Nombre d'incidents")
axes[1].legend(title="Groupe")

plt.tight_layout()
plt.show()

> **Observation clé :** Contrairement aux idées reçues, les **fausses alarmes représentent
> plus de 50% des interventions**, devant les services spéciaux et les incendies.
> La 1ère cause de fausse alarme : les alarmes automatiques (AFA).

### 2.2 Répartition par sous-type (StopCodeDescription)

In [ ]:
plt.figure(figsize=(12, 7))
incident['StopCodeDescription'].value_counts().sort_values(ascending=True).plot(
    kind='barh', color='steelblue')
plt.title("Répartition des sous-types d'incidents (2021–2025)")
plt.xlabel("Nombre d'incidents")
plt.tight_layout()
plt.show()

print("\nTop 5 sous-types :")
print(incident['StopCodeDescription'].value_counts(normalize=True)
      .head(5).mul(100).round(1).astype(str) + '%')

### 2.3 Variations selon l'heure de la journée

In [ ]:
plt.figure(figsize=(14, 6))
sns.countplot(data=incident, x='HourOfCall', hue='IncidentGroup',
              palette=PALETTE_CATEGORIES)
plt.title("Nombre d'incidents par heure et par type (2021–2025)")
plt.xlabel("Heure de l'appel")
plt.ylabel("Nombre d'incidents")
plt.xticks(rotation=0)
plt.legend(title="Groupe", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

> **Observation :** Les incidents sont **plus nombreux en soirée (18h–22h)** et la nuit.
> Les fausses alarmes dominent en journée (alarmes automatiques dans les bureaux/commerces).

### 2.4 Répartition géographique par Borough

In [ ]:
plt.figure(figsize=(10, 12))
incident['IncGeo_BoroughName'].value_counts().sort_values(ascending=True).plot(
    kind='barh', color='steelblue')
plt.title("Nombre d'incidents par Borough (2021–2025)")
plt.xlabel("Nombre d'incidents")
plt.tight_layout()
plt.show()

> **Observation :** **Westminster** concentre le plus d'interventions, suivi de Camden,
> Lambeth et Southwark. L'hypercentre londonien est la zone la plus sollicitée.

### 2.5 Analyse des variables continues

In [ ]:
variables_continues = [
    'FirstPumpArriving_AttendanceTime',
    'SecondPumpArriving_AttendanceTime',
    'NumStationsWithPumpsAttending',
    'NumPumpsAttending',
    'PumpCount',
    'PumpMinutesRounded',
    'Notional Cost (£)'
]

incident[variables_continues].describe().round(1)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14))

vars_plot = [
    ('NumStationsWithPumpsAttending', 'Nb casernes mobilisées par incident'),
    ('NumPumpsAttending',             'Nb camions ayant participé'),
    ('PumpCount',                     'Nb camions appelés (partis de caserne)'),
    ('PumpMinutesRounded',            'Temps cumulé sur les lieux (min)'),
    ('Notional Cost (£)',             'Coût estimatif par incident (£)'),
]

for i, (var, titre) in enumerate(vars_plot):
    row, col = divmod(i, 2)
    ax_hist = axes[row][col]
    sns.histplot(data=incident, x=var, ax=ax_hist, color='steelblue', kde=False)
    ax_hist.set_title(titre)
    ax_hist.set_xlabel("")

# Supprimer le 6ème subplot vide
fig.delaxes(axes[2][1])
plt.suptitle("Distribution des variables continues — Incidents", fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

> **Observations :**
> - La très grande majorité des incidents mobilise **1 seule caserne et 1 seul camion**
> - Les interventions impliquant 3+ casernes sont rares mais présentes → événements graves
> - Les coûts et temps cumulés présentent de nombreux outliers à investiguer

### 2.6 Détection des outliers — PumpCount

In [ ]:
# Détection IQR
q1 = incident['PumpCount'].quantile(0.25)
q3 = incident['PumpCount'].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr

outliers_pump = incident[incident['PumpCount'] > upper]
print(f"Nombre d'outliers PumpCount (seuil IQR) : {len(outliers_pump):,}")
print(f"Proportion : {len(outliers_pump)/len(incident)*100:.1f}%")

# Visualisation temporelle des outliers
plt.figure(figsize=(14, 5))
sns.scatterplot(data=outliers_pump, x='DateOfCall', y='PumpCount',
                color='red', alpha=0.5, s=20)
plt.title("Événements extrêmes (outliers PumpCount) — Distribution temporelle")
plt.xlabel("Date")
plt.ylabel("Nombre de camions mobilisés")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Analyse univariée — Mobilisations

### 3.1 Variables continues (temps d'intervention)

In [ ]:
vars_mob = ['TurnoutTimeSeconds', 'TravelTimeSeconds', 'AttendanceTimeSeconds']
mobilisation[vars_mob].describe().round(1)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14))

configs = [
    ('AttendanceTimeSeconds', "Temps total d'arrivée (secondes)", 360, '6 min (objectif véhicule 1)'),
    ('TurnoutTimeSeconds',    "Temps de préparation (secondes)",  None, None),
    ('TravelTimeSeconds',     "Temps de trajet (secondes)",       None, None),
]

for i, (var, titre, seuil, label_seuil) in enumerate(configs):
    ax_h = axes[i][0]
    ax_b = axes[i][1]

    sns.histplot(data=mobilisation, x=var, ax=ax_h, color='steelblue', kde=False)
    ax_h.set_title(f"{titre} — Histogramme")
    if seuil:
        ax_h.axvline(seuil, color='red', linestyle='--', label=label_seuil)
        ax_h.legend()

    sns.boxplot(data=mobilisation, x=var, ax=ax_b, color='lightblue')
    ax_b.set_title(f"{titre} — Boxplot")
    if seuil:
        ax_b.axvline(seuil, color='red', linestyle='--')

plt.suptitle("Distribution des temps de mobilisation (2021–2025)",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

> **Observations :**
> - Le temps médian de préparation (TurnoutTime) est ~71 secondes
> - Des valeurs extrêmes dépassent 20 minutes → événements exceptionnels à investiguer
> - La majorité des interventions se fait **sous le seuil légal de 6 minutes**

### 3.2 Codes de retard (DelayCode)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Distribution des codes de retard
delay_counts = mobilisation['DelayCode_Description'].value_counts()
delay_counts.sort_values(ascending=True).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title("Répartition des codes de retard (2021–2025)")
axes[0].set_xlabel("Nombre de mobilisations")

# Temps moyen d'intervention par code de retard
delay_time = (mobilisation.groupby('DelayCode_Description')['AttendanceTimeSeconds']
              .mean()
              .sort_values(ascending=False)
              .head(10))
delay_time_min = delay_time / 60
delay_time_min.sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title("Temps moyen d'intervention par code de retard (Top 10)")
axes[1].set_xlabel("Temps moyen (minutes)")
axes[1].axvline(6, color='green', linestyle='--', label='Objectif 6 min')
axes[1].legend()

plt.tight_layout()
plt.show()

> **Observation :** Les causes de retard les plus impactantes sont la **communication**
> (~9:11 min), l'**équipement** (~8:54 min) et la **disponibilité** (~8:15 min).
> Ces causes sont **maîtrisables** contrairement aux problèmes de circulation.

## 4. Analyse multivariée

### 4.1 Coûts par type d'incident et par année

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Coût total par type et par année
cost_type = incident.groupby(['CalYear', 'IncidentGroup'])['Notional Cost (£)'].sum().reset_index()
sns.barplot(data=cost_type, x='CalYear', y='Notional Cost (£)',
            hue='IncidentGroup', palette=PALETTE_CATEGORIES, ax=axes[0])
axes[0].set_title("Coût total par type d'incident et par année")
axes[0].set_xlabel("Année")
axes[0].legend(title="Groupe")

# Nb camions par type et par année
pump_type = incident.groupby(['CalYear', 'IncidentGroup'])['PumpCount'].sum().reset_index()
sns.lineplot(data=pump_type, x='CalYear', y='PumpCount',
             hue='IncidentGroup', marker='o', palette=PALETTE_CATEGORIES, ax=axes[1])
axes[1].set_title("Nombre de camions mobilisés par type d'incident")
axes[1].set_xlabel("Année")
axes[1].set_ylabel("Nombre de camions")
axes[1].legend(title="Groupe")

plt.tight_layout()
plt.show()

### 4.2 Localisation des incidents (carte de densité)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

groupes = incident['IncidentGroup'].unique()
colors = {'False Alarm': '#e74c3c', 'Special Service': '#3498db', 'Fire': '#f39c12'}

for i, groupe in enumerate(groupes):
    subset = incident[incident['IncidentGroup'] == groupe]
    axes[i].scatter(
        subset['Easting_rounded'], subset['Northing_rounded'],
        s=1, alpha=0.03, color=colors.get(groupe, 'grey'), linewidth=0
    )
    axes[i].set_xlim(500000, 560000)
    axes[i].set_ylim(150000, 210000)
    axes[i].set_title(f"Localisation — {groupe}")
    axes[i].set_xlabel("Easting")
    axes[i].set_ylabel("Northing")
    axes[i].set_aspect('equal')

plt.suptitle("Densité géographique des incidents par type (2021–2025)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Performance légale — Objectifs LFB

La LFB est soumise à des objectifs formalisés dans le **London Safety Plan** :
- 🎯 **Véhicule 1** : arrivée en ≤ 6 minutes (360 secondes)
- 🎯 **Véhicule 2** : arrivée en ≤ 8 minutes (480 secondes)

### 5.1 Création des indicateurs légaux

In [ ]:
incident['legal_1'] = np.where(
    incident['FirstPumpArriving_AttendanceTime'] <= 360, 1, 0)
incident['legal_2'] = np.where(
    (incident['SecondPumpArriving_AttendanceTime'] <= 480) &
    (incident['NumPumpsAttending'] >= 2), 1, 0)

print("Respect objectif véhicule 1 (≤ 6 min) :")
print(incident['legal_1'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')
print("\nRespect objectif véhicule 2 (≤ 8 min) :")
print(incident['legal_2'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

### 5.2 Évolution annuelle du respect des objectifs

In [ ]:
mediane_p1 = incident.groupby('CalYear')['FirstPumpArriving_AttendanceTime'].median() / 60
mediane_p2 = incident.groupby('CalYear')['SecondPumpArriving_AttendanceTime'].median() / 60

fig, (ax_g, ax_d) = plt.subplots(1, 2, figsize=(18, 7), sharey=False)

# --- GRAPHIQUE GAUCHE : Véhicule 1 ---
sns.histplot(data=incident, x='CalYear', hue='legal_1', multiple='dodge',
             discrete=True, shrink=0.8, palette=PALETTE_LEGAL, ax=ax_g)
ax_g.set_title("Véhicule 1 — Objectif ≤ 6 min")
ax_g.set_xlabel("Année")
ax_g.set_ylabel("Nombre d'incidents")
handles, _ = ax_g.get_legend_handles_labels()
ax_g.legend(handles, ['> 6 min (hors objectif)', '≤ 6 min (objectif atteint)'],
            title="Performance")

ax2_g = ax_g.twinx()
sns.lineplot(x=mediane_p1.index, y=mediane_p1.values,
             color='black', marker='o', linewidth=2, ax=ax2_g)
ax2_g.axhline(y=6, color='green', linestyle='--', alpha=0.7, label='Seuil 6 min')
ax2_g.set_ylabel("Temps médian (minutes)")
ax2_g.legend(loc='upper right')

# --- GRAPHIQUE DROIT : Véhicule 2 ---
sns.histplot(data=incident, x='CalYear', hue='legal_2', multiple='dodge',
             discrete=True, shrink=0.8, palette=PALETTE_LEGAL, ax=ax_d)
ax_d.set_title("Véhicule 2 — Objectif ≤ 8 min")
ax_d.set_xlabel("Année")
handles2, _ = ax_d.get_legend_handles_labels()
ax_d.legend(handles2, ['> 8 min (hors objectif)', '≤ 8 min (objectif atteint)'],
            title="Performance")

ax2_d = ax_d.twinx()
sns.lineplot(x=mediane_p2.index, y=mediane_p2.values,
             color='black', marker='o', linewidth=2, ax=ax2_d)
ax2_d.axhline(y=8, color='green', linestyle='--', alpha=0.7, label='Seuil 8 min')
ax2_d.set_ylabel("Temps médian (minutes)")
ax2_d.legend(loc='upper right')

plt.suptitle("Respect des objectifs légaux d'intervention (2021–2025)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> **Observation critique :** Le nombre d'interventions **hors objectif augmente chaque année**.
> Le temps médian d'intervention progresse également :
> - Véhicule 1 : 5:11 → 5:31 min (**+6,4%** sur 5 ans)
> - Véhicule 2 : 6:31 → 6:53 min (**+5,6%** sur 5 ans)
>
> La tendance est défavorable malgré un respect global des objectifs.

### 5.3 Analyse des retards par Borough

In [ ]:
retard_camion_1 = incident[incident['legal_1'] == 0]

total_an    = incident.groupby(['IncGeo_BoroughName', 'CalYear']).size()
retard_an   = retard_camion_1.groupby(['IncGeo_BoroughName', 'CalYear']).size()
pct_retard  = (retard_an / total_an * 100).fillna(0).unstack()

# Tri par taux moyen décroissant
pct_retard['Moyenne'] = pct_retard.mean(axis=1)
pct_retard = pct_retard.sort_values('Moyenne', ascending=False).drop(columns=['Moyenne'])

plt.figure(figsize=(12, 14))
sns.heatmap(pct_retard, cmap='Reds', annot=True, fmt=".1f",
            linewidths=.5, cbar_kws={'label': 'Taux de retard (%)'})
plt.title("Taux d'interventions hors objectif (> 6 min) par Borough et par année",
          pad=20, fontsize=13)
plt.xlabel("Année")
plt.ylabel("Borough")
plt.tight_layout()
plt.show()

> **Observation :** Les Boroughs périphériques (Havering, Bromley, Hillingdon)
> présentent les taux de retard les plus élevés et les plus persistants.
> Westminster, malgré son volume élevé d'incidents, maintient de meilleures performances.

### 5.4 Catégories d'incidents dans les retards

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Répartition des types dans les retards
ax = sns.histplot(data=retard_camion_1, x='CalYear', hue='IncidentGroup',
                  multiple='stack', discrete=True, shrink=0.8,
                  palette=PALETTE_CATEGORIES, ax=axes[0])
axes[0].set_title("Répartition des types d'incidents — Retards véhicule 1 (> 6 min)")
axes[0].set_xlabel("Année")
axes[0].set_ylabel("Nombre d'interventions en retard")
sns.move_legend(axes[0], "upper left", bbox_to_anchor=(1.01, 1), title="Catégorie")

# Top Boroughs avec le plus de retards
sns.countplot(data=retard_camion_1, y='IncGeo_BoroughName',
              order=retard_camion_1['IncGeo_BoroughName'].value_counts().index[:15],
              palette='Reds_r', ax=axes[1])
axes[1].set_title("Top 15 Boroughs — Retards véhicule 1")
axes[1].set_xlabel("Nombre de retards")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## Synthèse & Pistes d'action

### 🔍 Principaux enseignements

| Thème | Constat |
|---|---|
| **Volume** | +30% d'incidents sur 5 ans, budget en hausse de seulement +21% |
| **Type** | 50%+ de fausses alarmes — les alarmes automatiques (AFA) sont la 1ère cause |
| **Géographie** | Westminster et l'hypercentre concentrent le plus d'incidents |
| **Performance** | Objectifs légaux globalement respectés mais tendance défavorable |
| **Retards** | Boroughs périphériques les plus impactés — causes maîtrisables identifiées |

### 🎯 Pistes d'optimisation prioritaires

1. **Réduire les fausses alarmes AFA** → sensibilisation des gestionnaires de bâtiments,
   révision des protocoles d'alarmes automatiques
2. **Renforcer les casernes périphériques** en difficulté (Southall, Croydon, Finchley,
   Enfield, Dagenham) → audit matériel et communication
3. **Améliorer le remplissage des codes retard** (63% non renseignés en 2025)
   pour mieux orienter les actions correctives

---

**Pour une analyse interactive :** 👉 [Tableau de bord Power BI](https://app.powerbi.com/view?r=eyJrIjoiN2UyNTMxYTItZmY2Yy00OGQzLTk3ZjItOGFlZDRkNDY3NGE3IiwidCI6IjI3YThmZjFkLTc1YTAtNGFlNC1hNTg2LTMxN2NlOWVhYWZmYiJ9&pageName=4952a90622ed792a18d3)